In [ ]:
import yfinance as yf
import pandas as pd

tickers = [
    "BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD",
    "GC=F", "SI=F", "BZ=F", "NG=F", "HG=F", "ZC=F", "KC=F", "PA=F",
    "TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", "VTC", "JNK", "IBGL.L", "BTP.MI", "MUB",
    "AAPL", "MSFT", "AMZN", "JNJ", "JPM", "XOM", "PG", "TSLA", "UNH", "BRK-B",
    "SAN.MC", "ITX.MC", "IBE.MC", "MC.PA", "SAP.DE", "ASML.AS", "SIE.DE", "NESN.SW",
    "AZN.L", "HSBA.L",
    "2330.TW", "7203.T", "BABA", "TCEHY", "RELIANCE.NS", "VALE", "BHP"
]

rows = []

for ticker in tickers:
    info = yf.Ticker(ticker).info

    fcf = info.get("freeCashflow") # Flujo de caja libre del activo
    market_cap = info.get("marketCap") # Capitalización de mercado del activo

    if fcf and market_cap:
        fcf_yield = fcf / market_cap
    else:
        fcf_yield = None

    rows.append({
        "ticker": ticker, # Nombre del activo
        "asset_name": info.get("shortName"), # Nombre completo del activo
        "sector": info.get("sector"), # Sector al que pertenece el activo
        "exchange": info.get("exchange"), # Bolsa donde se cotiza el activo
        "currency": info.get("currency"), # Moneda en la que se cotiza el activo
        "beta": info.get("beta"), # Medida de la volatilidad del activo en comparación con el mercado
        "dividend_yield": info.get("dividendYield"), # Rendimiento por dividendo del activo
        "trailingpe": info.get("trailingPE"), # Relación precio-beneficio del activo basada en las ganancias pasadas
        "pricetobook":  info.get("priceToBook"), # Relación precio-valor contable del activo
        "fcf_yield": fcf_yield, # Rendimiento del flujo de caja libre del activo
        "revenuegrowth": info.get("revenueGrowth"), # Tasa de crecimiento de los ingresos del activo
        "earningsgrowth": info.get("earningsGrowth"), # Tasa de crecimiento de las ganancias del activo
        "forwardeps": info.get("forwardEps"), # Ganancias por acción proyectadas para el próximo año
        "payoutratio": info.get("payoutRatio")
    })

assets_df = pd.DataFrame(rows)


HTTP Error 404: Not Found{"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BTP.MI"}}}


In [2]:
assets_df["sector"].value_counts()

sector
Consumer Cyclical         6
Technology                5
Financial Services        4
Healthcare                3
Energy                    2
Consumer Defensive        2
Basic Materials           2
Utilities                 1
Industrials               1
Communication Services    1
Name: count, dtype: int64

In [4]:
assets_df["score_PER"] = 1 / assets_df["trailingpe"]
assets_df["score_PB"] = 1 / assets_df["pricetobook"]
assets_df["score_div"] = assets_df["dividend_yield"]
assets_df["score_fcf"] = assets_df["fcf_yield"]

assets_df["value_score"] = (
    assets_df["score_PER"] +
    assets_df["score_PB"] +
    assets_df["score_div"] +
    assets_df["score_fcf"]
)

assets_df = assets_df.drop(columns=["score_PER", "score_PB", "score_div", "score_fcf"])

## Las columnas con mayor value_score son las empresas mas infravaloradas



In [7]:
assets_df["rev_z"] = (assets_df["revenuegrowth"] - assets_df["revenuegrowth"].mean()) / assets_df["revenuegrowth"].std()
assets_df["earn_z"] = (assets_df["earningsgrowth"] - assets_df["earningsgrowth"].mean()) / assets_df["earningsgrowth"].std()

assets_df["growth_score"] = (
    assets_df["rev_z"] +
    assets_df["earn_z"]
)


In [10]:
assets_df["dividend_score"] = (
assets_df["dividend_yield"] *
(1 - assets_df["payoutratio"]) *
assets_df["fcf_yield"]
)


In [11]:
def asset_type(ticker):
    
    crypto = ["BTC-USD", "ETH-USD", "SOL-USD", "LINK-USD"]
    
    commodities = ["GC=F", "SI=F", "BZ=F", "NG=F", 
                   "HG=F", "ZC=F", "KC=F", "PA=F"]
    
    bonds = ["TLT", "IEF", "SHY", "TIP", "BNDX", "EMB", 
             "VTC", "JNK", "IBGL.L", "BTP.MI", "MUB"]
    
    if ticker in crypto:
        return "Crypto"
    
    elif ticker in commodities:
        return "Commodity"
    
    elif ticker in bonds:
        return "Bond"
    
    else:
        return "Equity"

assets_df["asset_type"] = assets_df["ticker"].apply(asset_type)

In [13]:
df_historico = pd.read_csv("./data/historical_assets.csv")

grupby_activo = df_historico.groupby("ticker")

returns = grupby_activo["Close"].pct_change()

volatility = returns.groupby(df_historico["ticker"]).std()

volatility_anual = volatility * (252 ** 0.5)

assets_df["volatility"] = assets_df["ticker"].map(volatility_anual)

In [14]:
def max_drawdown(prices):
    cummax = prices.cummax()
    drawdown = (prices - cummax) / cummax
    return drawdown.min()

prices_df = df_historico.pivot(index="date", columns="ticker", values="Close")

mdd = prices_df.apply(max_drawdown)

assets_df["max_drawdown"] = assets_df["ticker"].map(mdd)

In [17]:
def avg_return(prices, years):
    return (prices.iloc[-1] / prices.iloc[-252*years] - 1)

assets_df["avg_return_1y"] = prices_df.apply(avg_return, years=1)
assets_df["avg_return_3y"] = prices_df.apply(avg_return, years=3)
assets_df["avg_return_5y"] = prices_df.apply(avg_return, years=5)

In [18]:
def risk_category(vol):
    if vol < 0.10:
        return "Low"
    elif vol < 0.25:
        return "Medium"
    return "High"

assets_df["risk_category"] = assets_df["volatility"].apply(risk_category)

In [19]:
def investment_style(row):
    if row["asset_type"] == "Bond":
        return "Defensive"
    if row["dividend_yield"] and row["dividend_yield"] > 0.03:
        return "Income"
    return "Growth"

assets_df["investment_style"] = assets_df.apply(investment_style, axis=1)

In [20]:
assets_df.to_csv("data/assets_info.csv", index=False)

In [21]:
assets_df.head()

,ticker,asset_name,sector,exchange,currency,beta,dividend_yield,trailingpe,pricetobook,fcf_yield,...,growth_score,dividend_score,asset_type,volatility,max_drawdown,avg_return_1y,avg_return_3y,avg_return_5y,risk_category,investment_style
0,BTC-USD,Bitcoin USD,NaN,CCC,USD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Crypto,0.556579,-0.833990,NaN,NaN,NaN,High,Growth
1,ETH-USD,Ethereum USD,NaN,CCC,USD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Crypto,0.711614,-0.939625,NaN,NaN,NaN,High,Growth
2,SOL-USD,Solana USD,NaN,CCC,USD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Crypto,1.006502,-0.962725,NaN,NaN,NaN,High,Growth
3,LINK-USD,Chainlink USD,NaN,CCC,USD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Crypto,0.991540,-0.901929,NaN,NaN,NaN,High,Growth
4,GC=F,Gold Jun 26,NaN,CMX,USD,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Commodity,0.177782,-0.443638,NaN,NaN,NaN,Medium,Growth


In [ ]:
assets_buenos = assets_df.dropna(
    subset=["value_score", "growth_score", "dividend_score"]
)


(50, 27)